# Day 02 — Exercises
**Attempt before checking solutions in the last cell.**

Every exercise asks you to predict something about the Spark UI *before* running the code — this builds the habit of reasoning about execution, not just outcomes.


In [24]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("day02_exercises")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Ready. Spark UI: http://localhost:4040")


Ready. Spark UI: http://localhost:4040


---
### Exercise 1 — Predict the job/stage count

```python
orders = spark.createDataFrame([
    ("o1", "Toronto", 100),
    ("o2", "Calgary", 200),
    ("o3", "Toronto", 150),
    ("o4", "Montreal", 300),
    ("o5", "Calgary", 250),
], schema=["order_id", "city", "amount"])

result = (
    orders
    .filter(F.col("amount") > 100)
    .groupBy("city")
    .agg(F.sum("amount").alias("total"))
    .filter(F.col("total") > 150)
)

result.show()
result.count()
```

**Before running:** answer in comments —
- How many **jobs** will this create?
- How many **stages** in the first job?
- Will the second `filter()` (on `total`) cause an additional shuffle? Why or why not?

Then run it and verify in the UI.


In [25]:
# Your predictions:
# Jobs: 2
# Stages in job 1: 2
# Does the second filter shuffle? No

# Now run it:
orders = spark.createDataFrame([
    ("o1", "Toronto", 100),
    ("o2", "Calgary", 200),
    ("o3", "Toronto", 150),
    ("o4", "Montreal", 300),
    ("o5", "Calgary", 250),
    ("o4", "Edmonton", 300),
    ("o5", "Vancouver", 250),
], schema=["order_id", "city", "amount"])

result = (
    orders
    .filter(F.col("amount") > 100)
    .groupBy("city")
    .agg(F.sum("amount").alias("total"))
    .filter(F.col("total") > 150)
)

result.show()
result.count()


+---------+-----+
|     city|total|
+---------+-----+
|  Calgary|  450|
| Montreal|  300|
| Edmonton|  300|
|Vancouver|  250|
+---------+-----+



4

---
### Exercise 2 — Caching investigation

1. Take `orders` from Exercise 1.
2. Run `.count()` on it twice WITHOUT caching — note both job numbers in the UI.
3. Cache it, run `.count()` twice more.
4. In the Storage tab, note the size shown.
5. Answer: did caching reduce the **stage count** for the second post-cache job? Check the Stages tab for both post-cache jobs.


In [26]:
# Your code + observations here
orders.count()

orders.count()

orders.cache()

orders.count()

orders.count()


7

---
### Exercise 3 — Build a 2-shuffle query and find both Exchanges

Write a query on `orders` that has **two** wide transformations in sequence — for example, a `groupBy` followed by a `join` against another small DataFrame, or two different `groupBy` operations chained via an intermediate DataFrame.

Requirements:
- The plan (`.explain()`) should show **2 Exchange nodes**
- Verify this in both `.explain()` output AND the SQL tab's visual DAG

Hint: You could group orders by city to get totals, then join that back to a small "city region" lookup DataFrame, then group again by region.


In [27]:
# Your code here

data = [("Toronto", "Ontario"), ("Calgary", "Alberta"), ("Montreal", "Quebec"), ("Edmonton", "Alberta"), ("Vancouver", "BC")]

df_province = spark.createDataFrame(data, schema=["city", "province"])

df_city = orders.groupBy("city")\
    .agg((F.sum(F.col("amount")).alias("amount")))

df_city = df_city.join(df_province, df_city.city == df_province.city, "inner").drop(df_city["city"])

df_city = df_city.groupBy("province").agg((F.sum(F.col("amount"))).alias("amount"))

df_city.show()

+--------+------+
|province|amount|
+--------+------+
| Ontario|   250|
| Alberta|   750|
|  Quebec|   300|
|      BC|   250|
+--------+------+



In [28]:
df_city.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (16)
+- HashAggregate (15)
   +- Exchange (14)
      +- HashAggregate (13)
         +- Project (12)
            +- BroadcastHashJoin Inner BuildLeft (11)
               :- BroadcastExchange (8)
               :  +- HashAggregate (7)
               :     +- Exchange (6)
               :        +- HashAggregate (5)
               :           +- Filter (4)
               :              +- InMemoryTableScan (1)
               :                    +- InMemoryRelation (2)
               :                          +- * Scan ExistingRDD (3)
               +- Filter (10)
                  +- Scan ExistingRDD (9)


(1) InMemoryTableScan
Output [2]: [city#682, amount#683L]
Arguments: [city#682, amount#683L], [isnotnull(city#682)]

(2) InMemoryRelation
Arguments: [order_id#681, city#682, amount#683L], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan ExistingRDD [codegen id : 1]
Output [3]: [order_id#681, city#682, amount#683L]
Arguments: [order_i

---
### Exercise 4 — Conceptual: MLlib pattern matching

For each of the following, classify as **Transformer** or **Estimator**, and justify in one sentence:

1. `VectorAssembler`
2. `LinearRegression`
3. The object returned by `LinearRegression().fit(df)`
4. `StandardScaler` (before `.fit()`)
5. The object returned by `StandardScaler().fit(df)`


In [ ]:
# Run this cell last to free resources
spark.stop()
print("SparkSession stopped. Spark UI is no longer available.")


SparkSession stopped. Spark UI is no longer available.


In [ ]:
# Does this object need to learn something from the dataset?
# Yes → Estimator
# No → Transformer

# Your answers:
# 1. VectorAssembler: Transformer
# 2. LinearRegression: Estimator
# 3. LinearRegression().fit(df) result: Transformer
# 4. StandardScaler (unfitted): Estimator
# 5. StandardScaler().fit(df) result: Transformer


---
## Solutions


In [ ]:
# ── SOLUTION 1 ────────────────────────────────────────────────────────────────
# Jobs: 2 (one for .show(), one for .count())
# Stages in job 1: 2 (one shuffle at groupBy -> 2 stages)
# Second filter (on 'total'): does NOT cause an additional shuffle.
#   It operates on the ALREADY-AGGREGATED result, which lives in the same
#   post-shuffle stage. Filters on aggregated columns are applied in the
#   same stage as the aggregation (HashAggregate + Filter together).

orders = spark.createDataFrame([
    ("o1", "Toronto", 100),
    ("o2", "Calgary", 200),
    ("o3", "Toronto", 150),
    ("o4", "Montreal", 300),
    ("o5", "Calgary", 250),
], schema=["order_id", "city", "amount"])

result = (
    orders
    .filter(F.col("amount") > 100)
    .groupBy("city")
    .agg(F.sum("amount").alias("total"))
    .filter(F.col("total") > 150)
)
result.explain()
result.show()


# ── SOLUTION 2 ────────────────────────────────────────────────────────────────
print("Without cache:")
print(orders.count())  # job N
print(orders.count())  # job N+1 - both recompute from scratch, same stage count

orders.cache()
print("\nAfter cache (first call materialises):")
print(orders.count())  # job N+2 - computes AND caches

print("\nAfter cache (second call):")
print(orders.count())  # job N+3 - should show fewer/skipped stages in UI
# In the Stages tab for job N+3, you'll see the scan stage marked as
# "skipped" (greyed out) because the cached data was reused directly.
orders.unpersist()


# ── SOLUTION 3 ────────────────────────────────────────────────────────────────
city_region = spark.createDataFrame([
    ("Toronto", "East"),
    ("Calgary", "West"),
    ("Montreal", "East"),
], schema=["city", "region"])

# Shuffle 1: groupBy city
city_totals = orders.groupBy("city").agg(F.sum("amount").alias("total"))

# Shuffle 2: join requires shuffling both sides by join key (city)
joined = city_totals.join(city_region, on="city")

# Shuffle 3 (bonus): groupBy region on the joined result
region_totals = joined.groupBy("region").agg(F.sum("total").alias("region_total"))

region_totals.explain()  # look for multiple Exchange nodes
region_totals.show()
# Note: Spark's optimizer may sometimes reduce shuffles via broadcast joins
# for small DataFrames -- if city_region is small enough, Spark may
# broadcast it instead of shuffling. Check the plan for "BroadcastExchange"
# vs plain "Exchange" -- this is a preview of Day 7 (Joins)!


# ── SOLUTION 4 ────────────────────────────────────────────────────────────────
# 1. VectorAssembler   -> Transformer (has .transform(), no .fit() needed - stateless)
# 2. LinearRegression  -> Estimator (has .fit(), produces a Model)
# 3. fit() result       -> Transformer (LinearRegressionModel - has .transform()/predict)
# 4. StandardScaler (unfitted) -> Estimator (must .fit() to learn mean/std, THEN transform)
# 5. fit() result        -> Transformer (StandardScalerModel - applies learned scaling)


In [23]:
spark.stop()
